In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
import subprocess, sys

VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

In [5]:
pip_install(
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
)
print("W3D2 profiling pins installed")

installing: transformers==4.46.* accelerate==1.1.*
W3D2 profiling pins installed


In [6]:
import transformers
import accelerate

print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)

transformers: 4.46.3
accelerate: 1.1.1


In [7]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL)

tok.pad_token = tok.eos_token
tok.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
    device_map="cuda",
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [9]:
print("Model:", MODEL)
print("Device:", model.device)
print("Dtype:", model.dtype)

Model: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0
Dtype: torch.float16


In [10]:
import time
import threading
from transformers import TextIteratorStreamer

def prompt_of_len(n_tokens: int) -> str:
    base = "Explain the following in detail.\n"
    filler = "A data center serves many inference requests at once. " * 600
    ids = tok(base + filler)["input_ids"][:n_tokens]
    return tok.decode(ids)

def measure_stream(prompt: str, new_tokens: int = 128):
    enc = tok(prompt, return_tensors="pt").to("cuda")

    streamer = TextIteratorStreamer(
        tok,
        skip_prompt=True,
        skip_special_tokens=True
    )

    kwargs = dict(
        **enc,
        max_new_tokens=new_tokens,
        do_sample=False,
        streamer=streamer
    )

    th = threading.Thread(
        target=model.generate,
        kwargs=kwargs
    )

    t0 = time.time()
    th.start()

    stamps = []

    for _ in streamer:
        stamps.append(time.time())

    th.join()

    ttft = stamps[0] - t0

    if len(stamps) > 1:
        gaps = [
            b - a
            for a, b in zip(stamps, stamps[1:])
        ]
        tpot = sum(gaps) / len(gaps)
    else:
        tpot = 0.0

    total = stamps[-1] - t0

    return {
        "ttft_s": round(ttft, 4),
        "tpot_s": round(tpot, 4),
        "total_s": round(total, 4),
        "n_tokens": len(stamps)
    }

In [11]:
measure_stream(prompt_of_len(128), new_tokens=8)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


{'ttft_s': 1.4111, 'tpot_s': 0.0632, 'total_s': 1.917, 'n_tokens': 9}

In [12]:
ttft_by_len = {}

for n in [128, 512, 2048]:
    r = measure_stream(prompt_of_len(n))
    ttft_by_len[str(n)] = r["ttft_s"]
    print(n, r)

128 {'ttft_s': 0.044, 'tpot_s': 0.0502, 'total_s': 6.4739, 'n_tokens': 129}
512 {'ttft_s': 0.0642, 'tpot_s': 0.0386, 'total_s': 5.001, 'n_tokens': 129}
2048 {'ttft_s': 0.3104, 'tpot_s': 0.0336, 'total_s': 4.6131, 'n_tokens': 129}


In [13]:
import gc

def kv_formula_kb_per_token(
    layers=28,
    kv_heads=2,
    head_dim=128,
    dbytes=2
):
    return 2 * layers * kv_heads * head_dim * dbytes / 1024


def cache_bytes(pkv):
    """Bytes held by the KV cache itself."""
    if hasattr(pkv, "key_cache"):
        tensors = list(pkv.key_cache) + list(pkv.value_cache)
    else:
        tensors = [t for layer in pkv for t in layer]

    return sum(
        t.numel() * t.element_size()
        for t in tensors
    )

In [14]:
formula = kv_formula_kb_per_token()
print("formula KB/token:", formula)

formula KB/token: 28.0


In [15]:
def measure_kv(context: int, new_tokens: int = 256):
    torch.cuda.empty_cache()
    gc.collect()

    torch.cuda.reset_peak_memory_stats()

    enc = tok(
        prompt_of_len(context),
        return_tensors="pt"
    ).to("cuda")

    before = torch.cuda.memory_allocated()

    out = model.generate(
        **enc,
        max_new_tokens=new_tokens,
        do_sample=False,
        use_cache=True,
        return_dict_in_generate=True
    )

    torch.cuda.synchronize()

    peak = torch.cuda.max_memory_allocated()

    total_tokens = out.sequences.shape[1]

    return {
        "context": context,
        "total_tokens": int(total_tokens),
        "peak_kb_per_token": round(
            (peak - before) / total_tokens / 1024, 1
        ),
        "kv_kb_per_token": round(
            cache_bytes(out.past_key_values)
            / total_tokens
            / 1024,
            1
        ),
    }

In [16]:
kv_rows = [measure_kv(c) for c in [512, 2048, 4096]]

for r in kv_rows:
    print(r, " vs formula:", formula, "KB/token")

From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


{'context': 512, 'total_tokens': 768, 'peak_kb_per_token': 63.4, 'kv_kb_per_token': 28.0}  vs formula: 28.0 KB/token
{'context': 2048, 'total_tokens': 2304, 'peak_kb_per_token': 84.0, 'kv_kb_per_token': 28.0}  vs formula: 28.0 KB/token
{'context': 4096, 'total_tokens': 4352, 'peak_kb_per_token': 87.6, 'kv_kb_per_token': 28.0}  vs formula: 28.0 KB/token


In [17]:
import json

with open("kv_check.json", "w") as f:
    json.dump(
        {
            "formula_kb_per_token": formula,
            "measured_kb_per_token": kv_rows[-1]["kv_kb_per_token"],
            "peak_kb_per_token": kv_rows[-1]["peak_kb_per_token"],
        },
        f
    )

print("kv_check.json created")

kv_check.json created


In [18]:
QUEUE = [32, 32, 32, 256] * 6

print("Number of requests:", len(QUEUE))
print("Useful tokens:", sum(QUEUE))

Number of requests: 24
Useful tokens: 2112


In [19]:
def static_queue(
    batch: int,
    prompt: str = "Explain what an inference server does."
):
    """
    Static batching:
    A batch starts and no new request joins until
    every request in that batch has finished.
    """

    t0 = time.time()

    useful = 0
    slots = 0

    for i in range(0, len(QUEUE), batch):

        chunk = QUEUE[i:i + batch]

        # The batch runs until its slowest request.
        n = max(chunk)

        enc = tok(
            [prompt] * len(chunk),
            return_tensors="pt",
            padding=True
        ).to("cuda")

        model.generate(
            **enc,
            max_new_tokens=n,
            do_sample=False
        )

        useful += sum(chunk)

        # Number of token-slots actually decoded by GPU
        slots += n * len(chunk)

    dt = time.time() - t0

    return {
        "batch": batch,
        "wall_s": round(dt, 2),
        "tokens_per_s": round(useful / dt, 1),
        "slot_efficiency": round(useful / slots, 3)
    }

In [20]:
batch_rows = {}

for n in [1, 4, 8]:
    r = static_queue(n)

    batch_rows[str(n)] = r["tokens_per_s"]

    print(r)

{'batch': 1, 'wall_s': 60.59, 'tokens_per_s': 34.9, 'slot_efficiency': 1.0}
{'batch': 4, 'wall_s': 41.43, 'tokens_per_s': 51.0, 'slot_efficiency': 0.344}
{'batch': 8, 'wall_s': 21.62, 'tokens_per_s': 97.7, 'slot_efficiency': 0.344}


In [21]:
import json

baselines = {
    "model": MODEL,
    "dtype": "fp16",
    "ttft_s": ttft_by_len,
    "tpot_s": measure_stream(prompt_of_len(512))["tpot_s"],
    "batch": {
        k: v for k, v in batch_rows.items()
    },
}

with open("baselines.json", "w") as f:
    json.dump(baselines, f, indent=2)

print(json.dumps(baselines, indent=2))

{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "dtype": "fp16",
  "ttft_s": {
    "128": 0.044,
    "512": 0.0642,
    "2048": 0.3104
  },
  "tpot_s": 0.0329,
  "batch": {
    "1": 34.9,
    "4": 51.0,
    "8": 97.7
  }
}


In [22]:
from google.colab import files

files.download("baselines.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
# Green-check verifier for Lab W3D2 (inference anatomy).
# Paste this as the last cell of your day-2 notebook and run it. It reads
# baselines.json (the export you also downloaded) plus the KV measurement you
# saved to kv_check.json, and checks the schema and the sanity rules.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os

# Qwen2.5-1.5B KV cache: 2 x 28 layers x 2 kv_heads x 128 head_dim x 2 bytes.
KV_FORMULA_KB_PER_TOKEN = 2 * 28 * 2 * 128 * 2 / 1024  # 28.0


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def load_json(path: str):
    if not os.path.exists(path):
        fail(f"{path} not found")
    try:
        with open(path) as fh:
            return json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"{path} is not valid JSON: {exc}")


def main() -> None:
    b = load_json("baselines.json")

    # schema
    for key in ("model", "dtype", "ttft_s", "tpot_s", "batch"):
        if key not in b:
            fail(f"baselines.json missing key: {key}")
    if not isinstance(b["ttft_s"], dict) or not b["ttft_s"]:
        fail("ttft_s must be a non-empty object keyed by prompt length")
    if not isinstance(b["batch"], dict):
        fail("batch must be an object keyed by batch size")
    for size in ("1", "4", "8"):
        if size not in b["batch"]:
            fail(f"batch missing size {size}")

    # sanity 1: TTFT rises with prompt length - the day's actual physics.
    # Prefill reads the whole prompt before the first token, so a 2048-token
    # prompt must pay a visibly larger TTFT than a 128-token one (the reference
    # T4 run measured 0.037 s vs 0.312 s). A flat TTFT means prefill was not
    # measured (cached prompt, wrong timestamps, or a reused stream).
    tpot = b["tpot_s"]
    if not isinstance(tpot, (int, float)) or tpot <= 0:
        fail(f"tpot_s not a positive number: {tpot}")
    for plen, ttft in b["ttft_s"].items():
        if not isinstance(ttft, (int, float)) or ttft <= 0:
            fail(f"ttft_s[{plen}] not a positive number: {ttft}")
    if not b["ttft_s"]["2048"] > b["ttft_s"]["128"]:
        fail(f"TTFT did not rise with prompt length "
             f"(128: {b['ttft_s']['128']}, 2048: {b['ttft_s']['2048']}); "
             "prefill is not being measured")

    # sanity 2: batch-8 throughput beats batch-1
    b1, b8 = b["batch"]["1"], b["batch"]["8"]
    if not (isinstance(b1, (int, float)) and isinstance(b8, (int, float))):
        fail("batch tokens/s values must be numbers")
    if not b8 > b1:
        fail(f"batch-8 throughput ({b8}) not above batch-1 ({b1})")

    # sanity 3: measured KV within a factor of 2 of the formula
    kv = load_json("kv_check.json")
    measured = kv.get("measured_kb_per_token")
    if not isinstance(measured, (int, float)) or measured <= 0:
        fail("kv_check.json needs a positive measured_kb_per_token")
    lo, hi = KV_FORMULA_KB_PER_TOKEN / 2, KV_FORMULA_KB_PER_TOKEN * 2
    if not lo <= measured <= hi:
        fail(f"measured KV {measured} KB/token outside 2x of formula "
             f"{KV_FORMULA_KB_PER_TOKEN} (allowed {lo:.1f} to {hi:.1f})")

    print(f"ttft lengths: {sorted(b['ttft_s'])}, tpot_s: {tpot}")
    print(f"batch tokens/s 1/4/8: {b['batch']['1']}/{b['batch']['4']}/"
          f"{b['batch']['8']}")
    print(f"KV measured {measured} KB/token vs formula "
          f"{KV_FORMULA_KB_PER_TOKEN} KB/token")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


ttft lengths: ['128', '2048', '512'], tpot_s: 0.0329
batch tokens/s 1/4/8: 34.9/51.0/97.7
KV measured 28.0 KB/token vs formula 28.0 KB/token
GREEN CHECK: PASS
